# Clasificadores clásicos: Árboles de Decisión, SVM y Random Forest — Ejemplos

**Módulo 4 — Modelado y evaluación de enfoques supervisados · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`4.3_Arboles_de_Decision.pdf`](4.3_Arboles_de_Decision.pdf), [`4.4_SVMs.pdf`](4.4_SVMs.pdf) y [`4.5_RandomForest_Ensambles.pdf`](4.5_RandomForest_Ensambles.pdf), y lleva a código lo que allí se explica.

## Contenido

Para cada clasificador, dos ejemplos: uno **implementado desde cero** (para entender el mecanismo interno) y otro **usando `scikit-learn`** sobre un dataset real y vistoso.

1. **Árboles de decisión**
   1.1 Desde cero: reproducimos el ejemplo exacto de las diapositivas (¿quién juega ping-pong?) calculando entropía, ganancia de información e índice Gini a mano.
   1.2 Con `scikit-learn`: reconocimiento de **dígitos escritos a mano**.
2. **Support Vector Machines (SVM)**
   2.1 Desde cero: implementamos un SVM lineal por descenso de gradiente sobre la función de pérdida *hinge*.
   2.2 Con `scikit-learn`: el **truco del kernel** sobre dos círculos concéntricos — exactamente el ejemplo de "SVM no lineal" de las diapositivas.
3. **Random Forest**
   3.1 Desde cero: implementamos *bagging* (muestreo bootstrap + votación por mayoría) con el *Breast Cancer Wisconsin dataset*.
   3.2 Con `scikit-learn`: importancia de características sobre el *Wine dataset* — ¿qué propiedad química delata la variedad de uva?

> 💡 Este notebook **no requiere descargar nada ni tener cuenta de Kaggle**: *Digits*, *Breast Cancer* y *Wine* vienen incluidos en `scikit-learn`, y los demás ejemplos usan datos sintéticos generados en el propio notebook.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, load_wine, load_breast_cancer, make_circles, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

pd.set_option('display.max_columns', None)
rng = np.random.default_rng(42)

## 1. Árboles de decisión

### 1.1 Desde cero: ¿quién juega ping-pong?

Reproducimos el ejemplo exacto de las diapositivas: de **30 estudiantes** (10 niñas, 20 niños; 14 de clase IX, 16 de clase X), **15 juegan ping-pong** en su tiempo libre. Queremos encontrar qué variable —género o clase— separa mejor a quienes juegan de quienes no, usando **entropía / ganancia de información** y el **índice Gini**, calculados completamente a mano.

In [ ]:
# Reconstruimos el dataset por celda (género x clase) para que ambas marginales
# coincidan exactamente con las diapositivas: por género (F=10 con 2 jugadores,
# M=20 con 13) y por clase (IX=14 con 6 jugadores, X=16 con 9).
celdas = [
    ('F', 'IX', 4, 1),   # (género, clase, tamaño de la celda, cuántos juegan ping-pong)
    ('F', 'X',  6, 1),
    ('M', 'IX', 10, 5),
    ('M', 'X',  10, 8),
]
filas = []
for genero, clase, tamano, juegan in celdas:
    filas += [{'genero': genero, 'clase': clase, 'ping_pong': 1}] * juegan
    filas += [{'genero': genero, 'clase': clase, 'ping_pong': 0}] * (tamano - juegan)
estudiantes = pd.DataFrame(filas)

print('Total de estudiantes:', len(estudiantes))
print('Juegan ping-pong:', estudiantes['ping_pong'].sum())
print(pd.crosstab(estudiantes['genero'], estudiantes['ping_pong']))
print(pd.crosstab(estudiantes['clase'], estudiantes['ping_pong']))

In [ ]:
def entropia(etiquetas):
    """Entropía de Shannon de un conjunto de etiquetas."""
    etiquetas = np.asarray(etiquetas)
    if len(etiquetas) == 0:
        return 0.0
    _, conteos = np.unique(etiquetas, return_counts=True)
    probs = conteos / len(etiquetas)
    return -np.sum(probs * np.log2(probs))

def gini_pureza(etiquetas):
    """Gini (pureza) según la definición de las diapositivas: suma de p^2 + q^2.
    Entre más alto, más homogéneo (puro) es el grupo."""
    etiquetas = np.asarray(etiquetas)
    if len(etiquetas) == 0:
        return 0.0
    _, conteos = np.unique(etiquetas, return_counts=True)
    probs = conteos / len(etiquetas)
    return np.sum(probs ** 2)

def ganancia_informacion(df, columna_split, columna_objetivo='ping_pong'):
    entropia_padre = entropia(df[columna_objetivo])
    entropia_ponderada = sum(
        (len(grupo) / len(df)) * entropia(grupo[columna_objetivo])
        for _, grupo in df.groupby(columna_split)
    )
    return entropia_padre - entropia_ponderada, entropia_ponderada

def gini_split(df, columna_split, columna_objetivo='ping_pong'):
    return sum(
        (len(grupo) / len(df)) * gini_pureza(grupo[columna_objetivo])
        for _, grupo in df.groupby(columna_split)
    )

print(f"Entropía del nodo principal: {entropia(estudiantes['ping_pong']):.2f}  (15 de 30 juegan -> máxima incertidumbre = 1.0)")

In [ ]:
ig_genero, h_genero = ganancia_informacion(estudiantes, 'genero')
ig_clase, h_clase = ganancia_informacion(estudiantes, 'clase')

print(f"División por género: entropía ponderada = {h_genero:.2f}, ganancia de información = {ig_genero:.2f}")
print(f"División por clase:  entropía ponderada = {h_clase:.2f}, ganancia de información = {ig_clase:.2f}")

gini_genero = gini_split(estudiantes, 'genero')
gini_clase = gini_split(estudiantes, 'clase')
print(f"\nGini (pureza) ponderado por género: {gini_genero:.2f}")
print(f"Gini (pureza) ponderado por clase:  {gini_clase:.2f}")

assert ig_genero > ig_clase and gini_genero > gini_clase
print(f"\n-> GÉNERO gana con ambos criterios: mayor ganancia de información ({ig_genero:.2f} > {ig_clase:.2f})"
      f" y mayor Gini ({gini_genero:.2f} > {gini_clase:.2f}).")

Los números coinciden exactamente con las diapositivas: entropía ponderada 0.86 (género) vs. 0.99 (clase), ganancia de información 0.14 vs. 0.01, y Gini 0.59 vs. 0.51. **Género** separa mejor a quienes juegan ping-pong de quienes no — sería el primer nodo de decisión de nuestro árbol.

Verifiquemos construyendo ese primer nivel del árbol (un *stump*, profundidad 1) con `scikit-learn`, y confirmemos que elige la misma variable.

In [ ]:
X_stump = pd.get_dummies(estudiantes[['genero', 'clase']], drop_first=True).astype(float)
y_stump = estudiantes['ping_pong'].values

arbol_stump = DecisionTreeClassifier(max_depth=1, criterion='entropy', random_state=0).fit(X_stump, y_stump)
variable_elegida = X_stump.columns[arbol_stump.tree_.feature[0]]
print(f'Variable usada por scikit-learn para el primer split: {variable_elegida}')

fig, ax = plt.subplots(figsize=(6, 4))
plot_tree(arbol_stump, feature_names=X_stump.columns, class_names=['No juega', 'Juega'],
          filled=True, ax=ax)
plt.tight_layout()
plt.show()

### 1.2 Con `scikit-learn`: reconocimiento de dígitos escritos a mano

Ahora un ejemplo más vistoso: el dataset **Digits** de `scikit-learn` contiene 1797 imágenes de 8×8 píxeles de dígitos manuscritos (0-9). Entrenamos un árbol de decisión para que aprenda a reconocerlos.

In [ ]:
digits = load_digits()
Xd, yd = digits.data, digits.target
print(f'{Xd.shape[0]} imágenes, cada una de {Xd.shape[1]} píxeles (8x8), {len(np.unique(yd))} clases (dígitos 0-9)')

fig, axes = plt.subplots(1, 8, figsize=(10, 1.5))
for ax, img, etiqueta in zip(axes, digits.images[:8], digits.target[:8]):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(str(etiqueta))
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.3, random_state=42, stratify=yd)

arbol_digitos = DecisionTreeClassifier(max_depth=8, random_state=0).fit(Xd_tr, yd_tr)
acc_arbol = arbol_digitos.score(Xd_te, yd_te)
print(f'Accuracy del árbol (profundidad 8) reconociendo dígitos de prueba: {acc_arbol:.3f}')

In [ ]:
# Veamos 10 ejemplos del conjunto de prueba, con la predicción del árbol
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
idx_muestra = rng.choice(len(Xd_te), 10, replace=False)
pred_muestra = arbol_digitos.predict(Xd_te[idx_muestra])

for ax, i, pred in zip(axes.ravel(), idx_muestra, pred_muestra):
    ax.imshow(Xd_te[i].reshape(8, 8), cmap='gray_r')
    color = 'green' if pred == yd_te[i] else 'red'
    ax.set_title(f'real={yd_te[i]}  pred={pred}', color=color, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Los errores (en rojo, si aparecen) suelen ser dígitos que un humano también confundiría — por ejemplo, un 1 escrito con un trazo que se parece a un 8. Como vimos en el notebook de Módulo 4.2, podríamos repetir este experimento variando `max_depth` para observar el mismo patrón de *underfitting*/*overfitting* que vimos con K-NN.

## 2. Support Vector Machines (SVM)

### 2.1 Desde cero: SVM lineal por descenso de gradiente

Las diapositivas plantean el entrenamiento de un SVM como el problema de optimización:

$$\min_{\mathbf{w}, b} \frac{\|\mathbf{w}\|^2}{2} + C\sum_i \xi_i \quad \text{sujeto a} \quad y_i(\mathbf{w}\cdot\mathbf{x}_i + b) \geq 1 - \xi_i$$

Una forma equivalente y más fácil de programar es minimizar directamente la **pérdida *hinge* regularizada**:

$$L(\mathbf{w}, b) = \lambda\|\mathbf{w}\|^2 + \frac{1}{n}\sum_i \max(0,\ 1 - y_i(\mathbf{w}\cdot\mathbf{x}_i + b))$$

Implementamos su descenso de gradiente (estocástico, un ejemplo a la vez) desde cero.

In [ ]:
X_svm_toy, y_svm_toy = make_blobs(n_samples=60, centers=2, cluster_std=1.0, random_state=6)
y_pm1 = np.where(y_svm_toy == 0, -1, 1)  # SVM usa etiquetas {-1, 1}, no {0, 1}

def entrenar_svm_desde_cero(X, y, lr=0.001, lam=0.01, epochs=2000):
    """Descenso de gradiente estocástico sobre la pérdida hinge regularizada."""
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    for _ in range(epochs):
        for i in range(n):
            margen = y[i] * (np.dot(X[i], w) + b)
            if margen >= 1:
                # ejemplo ya bien clasificado y fuera del margen: solo regularizamos
                w -= lr * (2 * lam * w)
            else:
                # ejemplo dentro del margen o mal clasificado: también corregimos con el error
                w -= lr * (2 * lam * w - y[i] * X[i])
                b -= lr * (-y[i])
    return w, b

w_manual, b_manual = entrenar_svm_desde_cero(X_svm_toy, y_pm1)
pred_manual = np.sign(X_svm_toy @ w_manual + b_manual)
print(f'Accuracy del SVM desde cero: {accuracy_score(y_pm1, pred_manual):.3f}')

In [ ]:
svc_lineal = SVC(kernel='linear', C=1.0).fit(X_svm_toy, y_pm1)
print(f"Accuracy de SVC(kernel='linear') de scikit-learn: {svc_lineal.score(X_svm_toy, y_pm1):.3f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_svm_toy[:, 0], X_svm_toy[:, 1], c=y_pm1, cmap='bwr', s=25)

xx = np.linspace(X_svm_toy[:, 0].min(), X_svm_toy[:, 0].max(), 50)
yy_manual = -(w_manual[0] * xx + b_manual) / w_manual[1]
ax.plot(xx, yy_manual, 'k-', linewidth=2, label='Frontera (desde cero)')

w_sk, b_sk = svc_lineal.coef_[0], svc_lineal.intercept_[0]
yy_sklearn = -(w_sk[0] * xx + b_sk) / w_sk[1]
ax.plot(xx, yy_sklearn, 'g--', linewidth=2, label='Frontera (scikit-learn)')

ax.legend()
ax.set_title('SVM lineal: implementación propia vs. scikit-learn')
plt.tight_layout()
plt.show()

Ambas fronteras de decisión separan perfectamente las dos clases y quedan muy cerca una de la otra — nuestra implementación con descenso de gradiente converge a una solución casi idéntica a la del solver de `scikit-learn` (que resuelve el problema de optimización de forma exacta).

### 2.2 Con `scikit-learn`: el truco del kernel sobre círculos concéntricos

Este es **exactamente** el ejemplo de la diapositiva "SVM no lineal": una clase forma un círculo interior y la otra lo rodea completamente. **Ningún hiperplano (línea recta) puede separar estas dos clases.**

In [ ]:
X_circ, y_circ = make_circles(n_samples=200, noise=0.08, factor=0.4, random_state=0)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_circ, y_circ, test_size=0.3, random_state=42, stratify=y_circ)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap='bwr', s=20, edgecolors='k', linewidths=0.3)
ax.set_title('Dos clases no separables linealmente')
plt.tight_layout()
plt.show()

In [ ]:
svc_lin = SVC(kernel='linear').fit(Xc_tr, yc_tr)
svc_rbf = SVC(kernel='rbf', gamma='scale').fit(Xc_tr, yc_tr)

print(f"SVC(kernel='linear'): accuracy en test = {svc_lin.score(Xc_te, yc_te):.3f}")
print(f"SVC(kernel='rbf'):    accuracy en test = {svc_rbf.score(Xc_te, yc_te):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))

for ax, modelo, nombre in zip(axes, [svc_lin, svc_rbf], ['Kernel lineal (falla)', 'Kernel RBF (funciona)']):
    Z = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap='bwr', s=15, edgecolors='k', linewidths=0.3)
    ax.set_title(nombre)

plt.tight_layout()
plt.show()

El kernel **lineal** solo puede trazar una línea recta, así que su accuracy apenas supera el azar (~50%). El kernel **RBF** (gaussiano) mapea implícitamente los datos a un espacio de mayor dimensión donde sí son separables —el "truco del kernel" de las diapositivas— y recupera prácticamente el círculo real con una frontera de decisión curva.

## 3. Random Forest

### 3.1 Desde cero: *bagging* con el *Breast Cancer Wisconsin dataset*

Las diapositivas describen el mecanismo de Random Forest en cuatro pasos: (1) **bootstrap** —muestras con reemplazo del conjunto de entrenamiento—, (2) un árbol de decisión por cada muestra, (3) cada árbol predice, y (4) se combina por **votación por mayoría**. Implementamos exactamente ese mecanismo desde cero, usando `DecisionTreeClassifier` como el "ladrillo" de cada árbol individual (lo que programamos nosotros es el *ensamble*, no el árbol en sí — ya lo hicimos en la sección 1).

In [ ]:
cancer = load_breast_cancer()
Xb, yb = cancer.data, cancer.target
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=42, stratify=yb)

def bosque_desde_cero(X, y, n_arboles=15, profundidad=3, semilla=0):
    """Bagging: cada árbol se entrena sobre una muestra bootstrap distinta."""
    gen = np.random.default_rng(semilla)
    n = len(X)
    arboles = []
    for i in range(n_arboles):
        idx_boot = gen.integers(0, n, size=n)  # muestreo CON reemplazo
        arbol = DecisionTreeClassifier(max_depth=profundidad, random_state=i)
        arbol.fit(X[idx_boot], y[idx_boot])
        arboles.append(arbol)
    return arboles

def predecir_bosque(arboles, X):
    """Agregación: cada árbol vota, y gana la clase mayoritaria."""
    votos = np.array([arbol.predict(X) for arbol in arboles])
    return np.apply_along_axis(lambda v: np.bincount(v).argmax(), axis=0, arr=votos)

bosque_manual = bosque_desde_cero(Xb_tr, yb_tr, n_arboles=15, profundidad=3)
pred_bosque = predecir_bosque(bosque_manual, Xb_te)
acc_bosque_manual = accuracy_score(yb_te, pred_bosque)
print(f'Accuracy del bosque desde cero (15 árboles): {acc_bosque_manual:.3f}')

In [ ]:
# Comparemos contra un único árbol (sin ensamble) y contra RandomForestClassifier de sklearn
arbol_unico = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xb_tr, yb_tr)
acc_unico = arbol_unico.score(Xb_te, yb_te)

rf_sklearn = RandomForestClassifier(n_estimators=15, max_depth=3, random_state=0).fit(Xb_tr, yb_tr)
acc_rf_sklearn = rf_sklearn.score(Xb_te, yb_te)

print(f'Accuracy de un único árbol:              {acc_unico:.3f}')
print(f'Accuracy del bosque desde cero (bagging): {acc_bosque_manual:.3f}')
print(f'Accuracy de RandomForestClassifier:       {acc_rf_sklearn:.3f}')

Tal como explican las diapositivas, el bosque reduce la varianza del modelo: aunque cada árbol individual se entrena sobre una muestra distinta (y podría sobreajustarse a ella), **combinar sus votos produce un modelo más robusto** que un único árbol. Nuestra versión desde cero, con el mismo mecanismo de bootstrap + votación, iguala de cerca a `RandomForestClassifier`de `scikit-learn` (que además aleatoriza qué variables considera en cada división, un refinamiento extra que no implementamos aquí).

### 3.2 Con `scikit-learn`: ¿qué distingue a una variedad de uva? (importancia de características)

Cerramos con el **Wine dataset**: 178 muestras de vino de tres cultivares distintos, descritas por 13 propiedades químicas. Entrenamos un Random Forest para clasificar el cultivar y le preguntamos —como en la diapositiva "Random Forest - Importancia de la característica"— **cuáles propiedades usa más para decidir**.

In [ ]:
wine = load_wine()
Xw, yw = wine.data, wine.target
print(f'{Xw.shape[0]} muestras de vino, {Xw.shape[1]} propiedades químicas, {len(wine.target_names)} cultivares: {list(wine.target_names)}')

Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.3, random_state=42, stratify=yw)

rf_wine = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xw_tr, yw_tr)
print(f'Accuracy del Random Forest clasificando el cultivar: {rf_wine.score(Xw_te, yw_te):.3f}')

In [ ]:
importancias = pd.Series(rf_wine.feature_importances_, index=wine.feature_names).sort_values(ascending=False)
print('Las 5 propiedades químicas más importantes para distinguir el cultivar:')
print(importancias.head(5).round(3))

fig, ax = plt.subplots(figsize=(7, 5))
importancias.head(8).sort_values().plot.barh(ax=ax, color='teal')
ax.set_xlabel('Importancia (promedio sobre los 200 árboles del bosque)')
ax.set_title('¿Qué propiedad química delata la variedad de uva?')
plt.tight_layout()
plt.show()

Random Forest calcula esto exactamente como describen las diapositivas: para cada división en cada árbol, mide cuánto disminuyó la impureza (Gini) gracias a esa variable, y promedia esa contribución sobre los 200 árboles del bosque. Con una accuracy casi perfecta y un ranking de importancias interpretable, este es un ejemplo de cómo Random Forest sirve no solo para predecir, sino también para **entender qué variables importan**.

---

## Actividad abierta para discutir en clase

1. En la sección 1.1, ¿qué pasaría con la ganancia de información si, en vez de "género", tuviéramos una variable con un valor distinto para cada estudiante (como un ID único)? ¿Por qué esa variable nunca debería usarse para dividir un árbol, aunque tuviera ganancia de información alta?
2. En la sección 2.2, el kernel RBF tiene un hiperparámetro `gamma`. Prueben con valores muy pequeños y muy grandes de `gamma` en el código y observen qué le pasa a la frontera de decisión. ¿En qué se parece esto a la discusión de *underfitting*/*overfitting* del notebook 4.2?
3. En la sección 3.1, aumenten `n_arboles` en `bosque_desde_cero` (por ejemplo, a 50 o 100) y vuelvan a comparar contra un único árbol. ¿La ventaja del bosque crece, se mantiene o desaparece?
4. De los tres clasificadores de este notebook (árbol, SVM, Random Forest), ¿cuál escogerían si lo más importante fuera poder **explicarle a un no-experto** por qué el modelo tomó una decisión? ¿Y si lo más importante fuera la **precisión**, sin importar la interpretabilidad?